# 01. 표 데이터 전처리·모델링 (분류/회귀)

**사용법**: KAMP에서 내려받은 표 형태 데이터를 `practice/data/`에 넣고 아래 CONFIG를 수정합니다. 파일이 없으면 합성 데모 데이터로 흐름만 확인합니다.

**시험 흐름**: EDA → 베이스라인 csv 먼저 저장 → 특징 추가/모델 개선 → 재저장.

In [ ]:
import sys; sys.path.append("../common")
from utils import *
import numpy as np, pandas as pd
seed_everything()

## CONFIG

In [ ]:
DATA_PATH = "../data/train.csv"     # KAMP 데이터 파일명으로 변경
TEST_PATH = "../data/test.csv"      # 없으면 None
TARGET    = "target"                # 타깃 컬럼명
ID_COL    = None                    # 제출에 필요한 ID 컬럼(없으면 None)
TASK      = "classification"        # "classification" | "regression"
OUT_PATH  = "../outputs/submission.csv"

In [ ]:
import os
from sklearn.datasets import make_classification, make_regression
if os.path.exists(DATA_PATH):
    df = read_csv_auto(DATA_PATH)
    test = read_csv_auto(TEST_PATH) if TEST_PATH and os.path.exists(TEST_PATH) else None
else:
    print("⚠ 데이터 없음 → 합성 데모 데이터 사용 (실제 KAMP 데이터로 교체하세요)")
    X, y = make_classification(2000, 12, n_informative=6, weights=[.8, .2], random_state=0)
    df = pd.DataFrame(X, columns=[f"f{i}" for i in range(12)]); df[TARGET] = y
    df["cat"] = np.random.choice(list("abc"), len(df))
    df.loc[df.sample(frac=.05).index, "f1"] = np.nan
    test = df.drop(columns=TARGET).sample(300, random_state=1).reset_index(drop=True)
quick_eda(df, TARGET)

## 전처리 + 교차검증 (파이프라인으로 누수 방지)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

feats = [c for c in df.columns if c not in (TARGET, ID_COL)]
num = df[feats].select_dtypes("number").columns.tolist()
cat = [c for c in feats if c not in num]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
if TASK == "classification":
    model = HistGradientBoostingClassifier(random_state=42); cv = StratifiedKFold(5, shuffle=True, random_state=42); scoring = "f1_macro"
else:
    model = HistGradientBoostingRegressor(random_state=42); cv = KFold(5, shuffle=True, random_state=42); scoring = "neg_root_mean_squared_error"
pipe = Pipeline([("pre", pre), ("m", model)])
with timer("cv"):
    s = cross_val_score(pipe, df[feats], df[TARGET], cv=cv, scoring=scoring)
print(scoring, s.mean().round(4), "±", s.std().round(4))

## 베이스라인 제출 파일 저장 → 이후 개선
(리더보드가 없으므로 CV 점수를 기준으로 판단)

In [ ]:
pipe.fit(df[feats], df[TARGET])
if test is not None:
    pred = pipe.predict(test[feats])
    sub = pd.DataFrame({TARGET: pred}) if ID_COL is None else pd.DataFrame({ID_COL: test[ID_COL], TARGET: pred})
    save_submission(sub, OUT_PATH)

## 개선 체크리스트
- 클래스 불균형: `class_weight`, 임계값 조정
- 특징: 비율/차이/집계, 범주 빈도 인코딩
- 모델: LightGBM/XGBoost, 앙상블(평균/스태킹)
- 하이퍼파라미터는 시간 대비 효율이 낮으면 생략